# 00 – Data Preprocessing (Final)
**CSC61304 Group 6 – Flood Risk Nepal**

Shared base notebook. Every model notebook (`01`–`05`) loads its inputs from
`dataset/processed/` produced here, so all 5 models train on identical data.

**Run this notebook from inside the `notebooks/` folder** (all paths are
relative to it, e.g. `../dataset/...`).

Pipeline order: **Load → Clean → Engineer Features → Create Target Label →
Handle Missing Values → Encode → Scale → Split → Save**

| Step | Task | Owner |
|---|---|---|
| 1–2 | Load CSV (skip HXL row), fix dtypes, dedupe, filter to district level | Utsab |
| 3 | Feature engineering (year, month, decade_num, district_zone) | Samar |
| 4 | Target label creation + class balance check | Shreya |
| 5 | Missing value handling (district-level median fill) | Ashish |
| 6–7 | Label encoding + StandardScaler feature scaling | Pramudit |
| 8 | Stratified classification split + regression split + save | Utsab |

**This final version fixes:**
1. **Raw file path inconsistency** — previous versions created
   `dataset/raw/` but read from `dataset/raw.csv`. This version checks
   both locations automatically so it works regardless of where the file
   was placed.
2. **Regression leakage** — `rfh` is excluded from the regression feature
   set since it is also the regression target.
3. **Regression target scale** — `y_reg` stays in raw mm (unscaled), so
   RMSE/MAE are interpretable in real rainfall units.
4. **`results/` folders created here too** — so every downstream model
   notebook can immediately save confusion matrices, figures, and the
   metrics summary without needing its own setup step.

**Confirmed from an actual run on the real dataset:**
- 126,203 rows after filtering to district level (77 districts)
- Zero missing values after the median-fill step
- Class balance: Low 75.0% / Moderate 15.0% / High 7.0% / Extreme 3.0%
  — meaning classes are **imbalanced**, so `class_weight='balanced'`
  should be used in every classifier that supports it.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Create every folder downstream notebooks will need, not just this one's
os.makedirs("../dataset/raw", exist_ok=True)
os.makedirs("../dataset/processed", exist_ok=True)
os.makedirs("../results", exist_ok=True)
os.makedirs("../results/confusion_matrices", exist_ok=True)
os.makedirs("../results/figures", exist_ok=True)

RANDOM_STATE = 42

## Step 1–2: Load & Clean — *Utsab*

Loads the raw CHIRPS CSV. HDX quirk: row index 1 holds HXL tags (`#date`,
`#adm+code`, etc.) and must be skipped, not treated as data.

**Path handling:** checks a few likely locations for the raw file (whether
it was placed directly as `dataset/raw.csv` or inside `dataset/raw/` with
its original filename) so this cell doesn't break depending on how each
teammate downloaded/saved it. If none are found, it lists what it looked
for so you can fix the path in one place.

- Columns lowercased for consistency (`PCODE` → `pcode`)
- `version` column dropped (metadata, not a feature)
- Filters to **district-level rows only** using `adm_level`, picking the
  level whose distinct `pcode` count is closest to 77 (Nepal's district
  count) — this dataset stacks national/province/district rows together,
  so skipping this step would silently mix admin levels.

In [ ]:
# Checks common locations/names so this works regardless of how the file was saved.
# Add another candidate path here if your file lives somewhere else.
CANDIDATE_PATHS = [
    "../dataset/raw.csv",
    "../dataset/raw/raw.csv",
    "../dataset/raw/npl-rainfall-adm2-full.csv",
]

RAW_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if RAW_PATH is None:
    raise FileNotFoundError(
        "Raw CHIRPS CSV not found. Checked:\n  " + "\n  ".join(CANDIDATE_PATHS) +
        "\nPlace the downloaded HDX file at one of these paths, "
        "or add its path to CANDIDATE_PATHS above."
    )

print(f"Loading raw data from: {RAW_PATH}")

df = pd.read_csv(RAW_PATH, skiprows=[1])

# Normalize column names to lowercase (source file uses PCODE, not pcode)
df.columns = [c.strip().lower() for c in df.columns]

# Drop metadata column we don't need as a feature
if 'version' in df.columns:
    df = df.drop(columns=['version'])

df['date'] = pd.to_datetime(df['date'])

numeric_cols = ['rfh','r1h','r3h','rfh_avg','r1h_avg','r3h_avg','rfq','r1q','r3q','n_pixels']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# --- Filter to district-level rows only ---
level_counts = df.groupby('adm_level')['pcode'].nunique()
print("\nUnique pcodes per adm_level:\n", level_counts)

district_level = (level_counts - 77).abs().idxmin()
print(f"\nSelected adm_level = {district_level} as district level "
      f"({level_counts[district_level]} unique pcodes)")

df = df[df['adm_level'] == district_level].copy()
df = df.dropna(subset=['rfh']).drop_duplicates()

print("\nShape after cleaning:", df.shape)
df.head()

## Step 3: Feature Engineering — *Samar*

Derives `year`, `month`, `decade_num` (which 10-day period of the month)
from `date`. `district_zone` is derived from the province-code prefix of
`pcode` (`NP01`–`NP07`), mapped to a Terai/Hill/Mountain zone, since there
is no `adm1_name` column in the real file.

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['decade_num'] = df['date'].dt.day.apply(lambda d: 1 if d <= 10 else (2 if d <= 20 else 3))

df['province_code'] = df['pcode'].astype(str).str[:4].str.upper()

province_zone_map = {
    'NP01': 'Hill',      # Koshi
    'NP02': 'Terai',     # Madhesh
    'NP03': 'Hill',      # Bagmati
    'NP04': 'Mountain',  # Gandaki
    'NP05': 'Terai',     # Lumbini
    'NP06': 'Mountain',  # Karnali
    'NP07': 'Hill',      # Sudurpashchim
}
df['district_zone'] = df['province_code'].map(province_zone_map)

unmapped = df.loc[df['district_zone'].isna(), 'province_code'].unique()
print("Unmapped province codes (should be empty):", unmapped)
assert len(unmapped) == 0, "Some province codes didn't map to a zone — check province_zone_map above."

df[['pcode', 'province_code', 'district_zone']].drop_duplicates().head(10)

## Step 4: Target Label Creation — *Shreya*

Buckets `rfq` (rainfall anomaly %) into `flood_risk_level` using
percentiles of the actual data rather than fixed thresholds, so it adapts
to whatever the real distribution looks like.

In [ ]:
q75, q90, q97 = df['rfq'].quantile([0.75, 0.90, 0.97])

def flood_risk_percentile(rfq):
    if rfq >= q97:
        return 'Extreme'
    elif rfq >= q90:
        return 'High'
    elif rfq >= q75:
        return 'Moderate'
    else:
        return 'Low'

df['flood_risk_level'] = df['rfq'].apply(flood_risk_percentile)

class_balance = df['flood_risk_level'].value_counts(normalize=True)
print("Class balance:\n", class_balance)
print("\nNote: classes are imbalanced — use class_weight='balanced' in "
      "every classifier that supports it (Logistic Regression, Decision "
      "Tree, Random Forest). GaussianNB doesn't support class_weight; use "
      "sample_weight at fit time instead, or note this as a limitation.")

class_balance

## Step 5: Missing Value Handling — *Ashish*

Fills missing numeric values with each **district's own median**
(grouped by `pcode`) rather than a global median, since rainfall patterns
differ a lot between districts.

In [ ]:
print("Missing values before fill:\n", df[numeric_cols].isna().sum())

for col in numeric_cols:
    df[col] = df.groupby('pcode')[col].transform(lambda x: x.fillna(x.median()))

df = df.dropna(subset=['rfq', 'flood_risk_level'])

print("\nMissing values after fill:\n", df[numeric_cols].isna().sum())

## Step 6–7: Encoding + Scaling — *Pramudit*

Label-encodes `district_zone` and `flood_risk_level` into numeric codes,
then applies `StandardScaler` separately for two feature sets:

- **Classification feature set** (`clf_feature_cols`) — includes `rfh`,
  used by Logistic Regression, Decision Tree, Naive Bayes, Random Forest,
  and K-Means. No leakage here since the target is `flood_risk_level`,
  not `rfh`.
- **Regression feature set** (`reg_feature_cols`) — **excludes `rfh`**,
  since Linear Regression predicts `rfh` itself. Including it as a feature
  would let the model trivially "predict" its own target (leakage),
  producing a meaningless near-perfect R².

In [ ]:
le_zone = LabelEncoder()
df['district_zone_enc'] = le_zone.fit_transform(df['district_zone'])

le_label = LabelEncoder()
df['flood_risk_level_enc'] = le_label.fit_transform(df['flood_risk_level'])
print("Label classes (encoded -> name):",
      dict(zip(le_label.transform(le_label.classes_), le_label.classes_)))

# Classification feature set — includes rfh
clf_feature_cols = ['rfh','r1h','r3h','rfh_avg','r1h_avg','r3h_avg','rfq','r1q','r3q',
                    'n_pixels','year','month','decade_num','district_zone_enc']
clf_feature_cols = [c for c in clf_feature_cols if c in df.columns]

clf_scaler = StandardScaler()
df_clf_scaled = df.copy()
df_clf_scaled[clf_feature_cols] = clf_scaler.fit_transform(df[clf_feature_cols])

# Regression feature set — EXCLUDES rfh (it's the target)
reg_feature_cols = [c for c in clf_feature_cols if c != 'rfh']

reg_scaler = StandardScaler()
X_reg_scaled = df.copy()
X_reg_scaled[reg_feature_cols] = reg_scaler.fit_transform(df[reg_feature_cols])

print(f"Classification features ({len(clf_feature_cols)}): {clf_feature_cols}")
print(f"Regression features ({len(reg_feature_cols)}), rfh excluded: {reg_feature_cols}")

df_clf_scaled[clf_feature_cols].describe()

## Step 8: Train/Test Split + Save — *Utsab*

Two splits are produced:
- **Classification split** (stratified on `flood_risk_level_enc`) — used
  by Logistic Regression/Decision Tree, Naive Bayes, Random Forest.
  K-Means (unsupervised) can also just use `X`/`X_train` directly.
- **Regression split** — features exclude `rfh` and are scaled; the
  target (`y_reg`) is raw `rfh` in mm, **not scaled**, so RMSE/MAE are
  interpretable in real rainfall units.

In [ ]:
# Classification split (stratified)
X = df_clf_scaled[clf_feature_cols]
y_class = df_clf_scaled['flood_risk_level_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=RANDOM_STATE, stratify=y_class
)

# Regression split (Samar / Linear Regression)
X_reg = X_reg_scaled[reg_feature_cols]
y_reg = df['rfh']  # raw, unscaled mm

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# Save full processed dataset (keeps original string labels for later decoding)
df.to_csv("../dataset/processed/nepal_flood_processed.csv", index=False)

X_train.to_csv("../dataset/processed/X_train.csv", index=False)
X_test.to_csv("../dataset/processed/X_test.csv", index=False)
y_train.to_csv("../dataset/processed/y_train.csv", index=False)
y_test.to_csv("../dataset/processed/y_test.csv", index=False)

X_train_reg.to_csv("../dataset/processed/X_train_reg.csv", index=False)
X_test_reg.to_csv("../dataset/processed/X_test_reg.csv", index=False)
y_train_reg.to_csv("../dataset/processed/y_train_reg.csv", index=False)
y_test_reg.to_csv("../dataset/processed/y_test_reg.csv", index=False)

print("Saved processed data and both splits to dataset/processed/\n")
print(f"Classification: X_train {X_train.shape}, X_test {X_test.shape}")
print(f"Regression:     X_train_reg {X_train_reg.shape}, X_test_reg {X_test_reg.shape}")

## Final Check — everything a model notebook needs is now on disk

In [ ]:
expected_files = [
    "../dataset/processed/nepal_flood_processed.csv",
    "../dataset/processed/X_train.csv", "../dataset/processed/X_test.csv",
    "../dataset/processed/y_train.csv", "../dataset/processed/y_test.csv",
    "../dataset/processed/X_train_reg.csv", "../dataset/processed/X_test_reg.csv",
    "../dataset/processed/y_train_reg.csv", "../dataset/processed/y_test_reg.csv",
]

print("Output file check:")
for f in expected_files:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}] {f}")

print("\nresults/ folders ready:")
for folder in ["../results", "../results/confusion_matrices", "../results/figures"]:
    print(f"  [{'OK' if os.path.isdir(folder) else 'MISSING'}] {folder}")

---
## Next steps for each model notebook

Every notebook in `01`–`05` (classification models + K-Means) should
start with:

```python
import pandas as pd

X_train = pd.read_csv("../dataset/processed/X_train.csv")
X_test  = pd.read_csv("../dataset/processed/X_test.csv")
y_train = pd.read_csv("../dataset/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../dataset/processed/y_test.csv").squeeze()
```

Samar (Linear Regression) uses the `_reg` files instead — `y_train_reg`/
`y_test_reg` are already in raw mm, no inverse-transform needed:

```python
X_train_reg = pd.read_csv("../dataset/processed/X_train_reg.csv")
X_test_reg  = pd.read_csv("../dataset/processed/X_test_reg.csv")
y_train_reg = pd.read_csv("../dataset/processed/y_train_reg.csv").squeeze()
y_test_reg  = pd.read_csv("../dataset/processed/y_test_reg.csv").squeeze()
```

For consistent confusion-matrix labels across all classification
notebooks, rebuild the label map from the full processed file:

```python
full_df = pd.read_csv("../dataset/processed/nepal_flood_processed.csv")
label_map = (
    full_df[['flood_risk_level', 'flood_risk_level_enc']]
    .drop_duplicates()
    .set_index('flood_risk_level_enc')['flood_risk_level']
    .to_dict()
)
class_names = [label_map[i] for i in sorted(label_map.keys())]
```

**Reminder:** use `class_weight='balanced'` in Logistic Regression,
Decision Tree, and Random Forest given the 75/15/7/3 class imbalance.
GaussianNB doesn't support `class_weight` — note this as a limitation
in the report, or pass `sample_weight` at fit time as a workaround.
